In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

In [ ]:
CANDIDATES = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]
DATA_DIR = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, 'train')):
        DATA_DIR = c
        break
if DATA_DIR is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train' in dirs:
            DATA_DIR = root
            break
if DATA_DIR is None:
    DATA_DIR = '../data'

TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')
print(f'DATA_DIR: {DATA_DIR}')

FORMATIONS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

In [ ]:
# ===== 診断: 訓練データで formation_tvt の精度を確認 =====
print('=== Training diagnostic: formation_tvt accuracy ===')
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
rmse_list = []
nan_count = 0

for f in train_files[:50]:  # 最初の50ウェルで確認
    hw = pd.read_csv(f)
    if not any(col in hw.columns for col in FORMATIONS):
        continue
    anchor    = hw[hw['TVT_input'].notna()]
    eval_zone = hw[hw['TVT_input'].isna() & hw['TVT'].notna()]
    if len(eval_zone) == 0 or len(anchor) == 0:
        continue
    last = anchor.iloc[-1]
    la_tvt  = float(last['TVT_input'])
    la_z    = float(last['Z'])
    la_ancc = float(last['ANCC'])
    if not np.isfinite(la_ancc) or la_ancc == 0.0:
        nan_count += 1
        continue
    b = la_tvt + la_z - la_ancc
    f_tvt = b - eval_zone['Z'].values + eval_zone['ANCC'].values
    valid = np.isfinite(f_tvt)
    if valid.sum() == 0:
        nan_count += 1
        continue
    rmse = np.sqrt(np.mean((eval_zone['TVT'].values[valid] - f_tvt[valid])**2))
    rmse_list.append(rmse)

print(f'Wells checked: {len(rmse_list)}, ANCC invalid: {nan_count}')
print(f'formation_tvt RMSE: mean={np.mean(rmse_list):.4f}, median={np.median(rmse_list):.4f}')
print(f'RMSE < 1ft: {sum(r < 1 for r in rmse_list)}/{len(rmse_list)} wells')

In [ ]:
# ===== 訓練データから formation KNN テーブルを構築 =====
# 隠しテストに未知ウェルがある場合に X/Y 近傍で補完するため
print('Building formation KNN table from training data...')
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
knn_rows = []
for f in train_files:
    try:
        hw = pd.read_csv(f, usecols=['MD', 'X', 'Y', 'Z', 'TVT_input'] + FORMATIONS)
    except Exception:
        continue
    if not any(c in hw.columns for c in FORMATIONS):
        continue
    anchor = hw[hw['TVT_input'].notna()]
    if len(anchor) == 0:
        continue
    last = anchor.iloc[-1]
    row = {'X': float(last['X']), 'Y': float(last['Y']),
           'Z': float(last['Z']), 'TVT': float(last['TVT_input'])}
    for col in FORMATIONS:
        row[col] = float(last[col]) if col in last.index else np.nan
    knn_rows.append(row)

knn_df = pd.DataFrame(knn_rows).dropna(subset=['ANCC'])
print(f'KNN table: {len(knn_df)} training wells with valid ANCC')


def lookup_formations_knn(x, y, k=5):
    """X/Y 座標から最近傍 k 件の formation 値を距離加重平均で返す"""
    dists = np.sqrt((knn_df['X'].values - x)**2 + (knn_df['Y'].values - y)**2)
    idx = np.argsort(dists)[:k]
    w = 1.0 / (dists[idx] + 1e-6)
    w /= w.sum()
    result = {}
    for col in FORMATIONS:
        result[col] = float(np.dot(w, knn_df[col].values[idx]))
    return result


# ===== 診断: テストウェルの formation 列 lookup を確認 =====
print('\n=== Test well formation lookup diagnostic ===')
sub = pd.read_csv(SAMPLE_SUB)
test_well_ids = sorted(sub['id'].str.rsplit('_', n=1).str[0].unique())
print(f'Test wells ({len(test_well_ids)}): {test_well_ids[:5]}...')

for well_id in test_well_ids[:3]:  # 最初の3件だけ表示
    print(f'\n--- {well_id} ---')
    hw_test = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))
    train_path = os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv')
    in_train = os.path.exists(train_path)
    print(f'In training: {in_train}, rows: {len(hw_test)}')

In [ ]:
# ===== formation_tvt のみで submission 作成 =====
print('\n=== Generating formation-only submission ===')
all_preds = {}
stats = {'train_lookup': 0, 'knn_lookup': 0, 'fallback': 0}

for well_id in test_well_ids:
    try:
        hw_test = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))

        # formation 列の取得方法を選択
        train_path = os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv')

        if os.path.exists(train_path):
            # 訓練データに同一ウェルあり → exact merge
            hw_tr = pd.read_csv(train_path)
            merged = hw_test.merge(hw_tr[['MD'] + FORMATIONS], on='MD', how='left')
            n_exact = merged['ANCC'].notna().sum()
            if n_exact >= len(hw_test) * 0.5:
                for col in FORMATIONS:
                    hw_test[col] = merged[col].interpolate(limit_direction='both').fillna(0)
                stats['train_lookup'] += 1
            else:
                # MD ずれ → nearest-neighbor
                train_md = hw_tr['MD'].values
                idx = np.searchsorted(train_md, hw_test['MD'].values).clip(0, len(train_md)-1)
                for col in FORMATIONS:
                    if col in hw_tr.columns:
                        hw_test[col] = hw_tr[col].values[idx]
                stats['train_lookup'] += 1
        elif not any(c in hw_test.columns for c in FORMATIONS):
            # 訓練データに未知ウェル → X/Y KNN で補完
            anchor_tmp = hw_test[hw_test['TVT_input'].notna()]
            if len(anchor_tmp) > 0:
                last_tmp = anchor_tmp.iloc[-1]
                knn_vals = lookup_formations_knn(float(last_tmp['X']), float(last_tmp['Y']))
                for col in FORMATIONS:
                    hw_test[col] = knn_vals[col]
                stats['knn_lookup'] += 1
            else:
                for col in FORMATIONS:
                    hw_test[col] = 0.0
                stats['fallback'] += 1

        # anchor / eval_zone
        anchor    = hw_test[hw_test['TVT_input'].notna()]
        eval_zone = hw_test[hw_test['TVT_input'].isna()]
        if len(anchor) == 0 or len(eval_zone) == 0:
            stats['fallback'] += 1
            continue

        last    = anchor.iloc[-1]
        la_tvt  = float(last['TVT_input'])
        la_z    = float(last['Z'])
        la_ancc = float(last.get('ANCC', 0.0))

        if not np.isfinite(la_ancc) or la_ancc == 0.0:
            f_tvt = np.full(len(eval_zone), la_tvt)
        else:
            b = la_tvt + la_z - la_ancc
            f_tvt = b - eval_zone['Z'].values + eval_zone['ANCC'].values
            f_tvt = np.where(np.isfinite(f_tvt), f_tvt, la_tvt)

        for idx_row, tvt in zip(eval_zone.index, f_tvt):
            all_preds[f'{well_id}_{idx_row}'] = tvt

    except Exception as e:
        print(f'ERROR on {well_id}: {e}')
        # 最終フォールバック: eval zone の TVT_input を anchor_tvt で埋める
        try:
            hw_test = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))
            anchor    = hw_test[hw_test['TVT_input'].notna()]
            eval_zone = hw_test[hw_test['TVT_input'].isna()]
            if len(anchor) > 0 and len(eval_zone) > 0:
                la_tvt = float(anchor.iloc[-1]['TVT_input'])
                for idx_row in eval_zone.index:
                    all_preds[f'{well_id}_{idx_row}'] = la_tvt
        except Exception:
            pass
        stats['fallback'] += 1

print(f'Lookup stats: {stats}')
sub['tvt'] = sub['id'].map(all_preds)
print(f'Null count: {sub["tvt"].isnull().sum()}')
print(sub.describe())
sub.to_csv('submission.csv', index=False)
print('\nSaved: submission.csv')